# Getting Started
This is a short demo for the interfaces for decoders defined in `bloqade-decoders` and how to use them.

We will cover two concrete implementations of the decoders: a lookup-table decoder (`TableDecoder`) that constructs a table from each detector pattern to the frequency of the observable corrections, and a most-likely error (MLE) decoder (`GurobiDecoder`) that solves for the most likely error that triggered the detector flips.

## Constructing decoders
To construct a decoder, you pass in the `stim.DetectorErrorModel`. Depending on the decoder, you can optionally pass in arguments to initialize the decoder.

In [87]:
# Define a stim detector error model used for decoding
import stim

demo_dem = stim.DetectorErrorModel("""
    error(0.09) D0 L0 L1
    error(0.1) D0 L1 L2
""")

In [88]:
# Construct a lookup-table decoder.
from bloqade.decoders import TableDecoder

lookup_table_decoder = TableDecoder(demo_dem)

In [89]:
# Construct a most-likely error decoder.
from bloqade.decoders import GurobiDecoder

mle_decoder = GurobiDecoder(demo_dem)

You can also optionally specify some initialization arguments that can be passed as keywords to initialize the decoders. For example, for the `TableDecoder`, you can specify the `num_shots` used to train it, and for the `GurobiDecoder`, you can specify the verbosity in logging (whether `verbose` is True).
> By default, the `TableDecoder` uses 10,000 shots for training, and the `GurobiDecoder` has `verbose` set to False.

In [90]:
lookup_table_decoder_1millionshots = TableDecoder(demo_dem, num_shots=1_000_000)

In [91]:
mle_decoder_verbose = GurobiDecoder(demo_dem, verbose=True)

## Performing decoding
Each decoder defines a `decode` method, which takes in a numpy array of detector bits. You can supply an array of bits in for one detector pattern, or you can supply a batch of detectors.

In [92]:
# Use numpy for defining some mock detector patterns
import numpy as np

In [93]:
lookup_table_correction = lookup_table_decoder_1millionshots.decode(detector_bits=np.array([True]))

In [94]:
# Returns the correction for L1 and L2 as those observable flips; the most frequently seen corrections.
lookup_table_correction

array([False,  True,  True])

In [95]:
mle_correction = mle_decoder.decode(detector_bits=np.array([True]))

In [96]:
# Returns the most likely error; in this case, the error that triggers D0, L1, and L2.
mle_correction

array([False,  True,  True])

In [97]:
# We can also get corrections in batches by supplying multiple detector patterns.
lookup_table_correction_batched = lookup_table_decoder_1millionshots.decode(detector_bits=np.array([
    [True],
    [False]
]))

In [98]:
lookup_table_correction_batched

array([[False,  True,  True],
       [False, False, False]])

In [99]:
mle_correction_batched = mle_decoder.decode(detector_bits=np.array([
    [True],
    [False],
]))

In [100]:
mle_correction_batched

array([[False,  True,  True],
       [False, False, False]])

## Obtaining Confidence from Decoding
Using the `decode_confidence(detectors)` method, you can additionally obtain a confidence value regarding how "confident" you are in your decoding. Understanding this confidence score will vary based on the decoder, but generally the scores will be in the interval `[0.0, 1.0]` where a higher value indicates a higher confidence in decoding.
> For the TableDecoder, the confidence for a given observable correction is computed by the fraction of shots seen for that correction divided by the total number of shots seen for that detector; for the GurobiDecoder, the confidence is the ratio of the probability of the most likely error and the second most likely error.

The inputs for `decode_confidence` is the same as `decode`. `decode_confidence` additionally returns a float or array of floats representing the confidence score for each detector.
> The default implementation of `decode_confidence` returns all corrections as equally confident (1.0).

In [103]:
lookup_table_correction_confidence = lookup_table_decoder_1millionshots.decode_confidence(detector_bits=np.array([True]))

In [ ]:
# The confidence here is roughly (0.1 / (0.1 + 0.09)).
lookup_table_correction_confidence

(array([False,  True,  True]), 0.528226135872927)

In [105]:
mle_correction_confidence = mle_decoder.decode_confidence(detector_bits=np.array([True]))

In [ ]:
# The confidence here is small due to the most likely and second most likely correction being similarly probable.
mle_correction_confidence

(array([False,  True,  True]), 0.05813953488372105)

In [107]:
lookup_table_correction_confidence_batch = lookup_table_decoder_1millionshots.decode_confidence(detector_bits=np.array([
    [True],
    [False]
]))

In [108]:
lookup_table_correction_confidence_batch

(array([[False,  True,  True],
        [False, False, False]]),
 array([0.52822614, 0.98899393]))

In [109]:
mle_correction_confidence_batch = mle_decoder.decode_confidence(detector_bits=np.array([
    [True],
    [False]
]))

In [110]:
mle_correction_confidence_batch

(array([[False,  True,  True],
        [False, False, False]]),
 array([0.05813953, 0.97826087]))